In [ ]:
'''
import polars as pl
import time

input_path = "dataset/*.parquet"  
output_file = "dataset/2010.parquet"

print(f"Start merging dataset in {input_path} ...")
start_time = time.time()

# Lazy Mode to scan all the dataset chunk and write with streaming(sink) to a single file.
# It can make sure that not all data chunk will be loaded into memory once.
pl.scan_parquet(input_path).sink_parquet(output_file)

end_time = time.time()
print(f"Done！File {output_file} Generated")
print(f"Time Used: {end_time - start_time:.2f} s")
'''

In [ ]:
import polars as pl
import glob
import time

# Make sure to place the monthly parquet files you want to merge into the 'raw_data' folder
input_files = glob.glob("dataset/yearly collections/GT10GB/*.parquet")  
output_file = "dataset/yearly collections/GT10GB/Res/2010_2011.parquet"

print(f"Found {len(input_files)} Parquet files. Starting schema processing and merging...")
start_time = time.time()

lazy_dfs = []
for file in input_files:
    # 1. Scan individual file (Lazy Mode)
    ldf = pl.scan_parquet(file)
    
    # Get the column names of the current file
    # Note: If you are using an older version of Polars, use `cols = ldf.columns` instead
    cols = ldf.collect_schema().names() 
    
    # 2. Force casting of problematic columns to a unified String type
    # If the file contains the 'rate_code' column, cast it to String to resolve conflicts
    if "rate_code" in cols:
        ldf = ldf.with_columns(pl.col("rate_code").cast(pl.String))
        
    # The NYC dataset sometimes has another column with similar schema issues. 
    # We preemptively handle it here to avoid crashes.
    if "store_and_fwd_flag" in cols:
        ldf = ldf.with_columns(pl.col("store_and_fwd_flag").cast(pl.String))
        
    lazy_dfs.append(ldf)

# 3. Lazily concatenate all files
print("Building the execution plan for concatenation...")
# use "diagonal_relaxed" mode to avoid merging failure caused by slight differences (e.g. Int32/64) in columns of different monthly data chunk
merged_lazy = pl.concat(lazy_dfs, how="diagonal_relaxed")

# 4. Trigger streaming write (Sink)
print("Executing streaming write (sink_parquet). This might take a moment, but memory usage will remain stable...")
merged_lazy.sink_parquet(output_file)

end_time = time.time()
print(f"Merge completed! Generated clean dataset: {output_file}")
print(f"Time elapsed: {end_time - start_time:.2f} seconds")

Found 12 Parquet files. Starting schema processing and merging...
Building the execution plan for concatenation...
Executing streaming write (sink_parquet). This might take a moment, but memory usage will remain stable...
Merge completed! Generated clean dataset: dataset/2011.parquet
Time elapsed: 25.57 seconds


In [7]:
import polars as pl
import os
import re
import time

folder_path = "dataset"

# 1. Fetch all parquet files (excluding files that are already merged to avoid infinite loops)
all_files = [f for f in os.listdir(folder_path) if f.endswith(".parquet") and "merged" not in f]

if not all_files:
    print(f"No valid parquet files found in '{folder_path}'")
else:
    # 2. Group files by their year extracted from the filename
    files_by_year = {}
    for file_name in all_files:
        # Extract the 4-digit year (e.g., "2010" from "yellow_tripdata_2010-12.parquet")
        match = re.search(r'(\d{4})', file_name)
        if match:
            year = match.group(1)
            full_path = os.path.join(folder_path, file_name)
            
            if year not in files_by_year:
                files_by_year[year] = []
            files_by_year[year].append(full_path)

    print(f"Found data for {len(files_by_year)} different years: {list(files_by_year.keys())}\n")

    # 3. Loop through each year and perform the merge
    for year, input_files in files_by_year.items():
        output_file = os.path.join(folder_path, f"{year}_merged.parquet")
        print(f"Processing Year {year}: Merging {len(input_files)} files into {output_file}...")
        start_time = time.time()

        lazy_dfs = []
        for file in input_files:
            ldf = pl.scan_parquet(file)
            cols = ldf.collect_schema().names() 
            
            # Resolve common NYC taxi schema drift issues by unifying types to String
            if "rate_code" in cols:
                ldf = ldf.with_columns(pl.col("rate_code").cast(pl.String))
            if "store_and_fwd_flag" in cols:
                ldf = ldf.with_columns(pl.col("store_and_fwd_flag").cast(pl.String))
                
            lazy_dfs.append(ldf)

        # Concatenate using 'diagonal_relaxed' to automatically handle missing/extra columns and type mismatches
        merged_lazy = pl.concat(lazy_dfs, how="diagonal_relaxed")
        
        # Execute streaming write
        merged_lazy.sink_parquet(output_file)

        end_time = time.time()
        print(f"Year {year} completed! Generated: {output_file}")
        print(f"Time elapsed for {year}: {end_time - start_time:.2f} seconds")
        print("-" * 50)
        
    print("All years have been successfully merged!")

Found data for 9 different years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']

Processing Year 2013: Merging 12 files into dataset\2013_merged.parquet...
Year 2013 completed! Generated: dataset\2013_merged.parquet
Time elapsed for 2013: 19.94 seconds
--------------------------------------------------
Processing Year 2014: Merging 12 files into dataset\2014_merged.parquet...
Year 2014 completed! Generated: dataset\2014_merged.parquet
Time elapsed for 2014: 20.32 seconds
--------------------------------------------------
Processing Year 2015: Merging 12 files into dataset\2015_merged.parquet...
Year 2015 completed! Generated: dataset\2015_merged.parquet
Time elapsed for 2015: 19.52 seconds
--------------------------------------------------
Processing Year 2016: Merging 12 files into dataset\2016_merged.parquet...
Year 2016 completed! Generated: dataset\2016_merged.parquet
Time elapsed for 2016: 19.31 seconds
--------------------------------------------------
